# Pipeline de Transcription et Sous-titrage

**Module :** 04-Audio-Applications  
**Niveau :** Applications  
**Technologies :** Whisper (local + API), pyannote (diarisation), pysrt, GPT (LLM)  
**VRAM estimee :** ~12 GB  
**Duree estimee :** 55 minutes  

## Objectifs d'Apprentissage

- [ ] Transcrire un fichier audio avec horodatage (timestamps)
- [ ] Mettre en place un pipeline de transcription batch (plusieurs fichiers)
- [ ] Comprendre la diarisation de locuteurs (identification des intervenants)
- [ ] Generer des sous-titres aux formats SRT et VTT
- [ ] Resumer automatiquement des reunions (transcription -> LLM -> resume)
- [ ] Evaluer la qualite de transcription et appliquer du post-traitement

## Prerequis

- Notebooks Foundation (01-2 Whisper STT, 01-4 Whisper Local) completes
- Cle API OpenAI configuree (`OPENAI_API_KEY` dans `.env`)
- faster-whisper installe (transcription locale)
- Comprehension du STT et des formats de sous-titres

**Navigation :** [Index](../README.md) | [<< Précédent](04-1-Educational-Audio-Content.ipynb) | [Suivant >>](04-3-Music-Composition-Workflow.ipynb)

In [1]:
# Parametres Papermill - JAMAIS modifier ce commentaire

# Configuration notebook
notebook_mode = "interactive"        # "interactive" ou "batch"
skip_widgets = False               # True pour mode batch MCP
debug_level = "INFO"

# Parametres transcription
whisper_model = "large-v3-turbo"   # Modele Whisper local
whisper_device = "cuda"            # "cuda" ou "cpu"
whisper_api_model = "whisper-1"    # Modele API OpenAI
default_language = "fr"            # Langue par defaut
llm_model = "gpt-5.6-luna"         # Modele pour le resume

# Configuration pipeline
use_local_whisper = False          # Desactive Whisper local pour validation (problèmes CUDA/CUBLAS)
generate_audio = True              # Generer des fichiers audio de test
save_output_files = True           # Sauvegarder les transcriptions


In [2]:
# Parameters
BATCH_MODE = "true"


L'environnement de travail est configure avec les paramètres Papermill. La cellule suivante charge les bibliotheques Python necessaires : traitement audio, appels API Whisper et utilitaires de formatage.

In [3]:
# Setup environnement et imports
import os
import sys
import json
import time
from pathlib import Path
from datetime import datetime, timedelta
from typing import Dict, List, Any, Optional, Tuple
from io import BytesIO
import logging

from IPython.display import Audio, display, HTML

# Resolution GENAI_ROOT
GENAI_ROOT = Path.cwd()
while GENAI_ROOT.name != 'GenAI' and len(GENAI_ROOT.parts) > 1:
    GENAI_ROOT = GENAI_ROOT.parent

HELPERS_PATH = GENAI_ROOT / 'shared' / 'helpers'
if HELPERS_PATH.exists():
    sys.path.insert(0, str(HELPERS_PATH.parent))
    try:
        from helpers.audio_helpers import (
            transcribe_openai, transcribe_local,
            synthesize_openai, get_audio_info, display_audio_bundle
        )
        print("Helpers audio importes")
    except ImportError as e:
        print(f"Helpers audio non disponibles - mode autonome : {e}")

# Repertoires
OUTPUT_DIR = GENAI_ROOT / 'outputs' / 'audio' / 'transcription'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Affichage des chemins : relatif au depot, jamais la racine machine du poste
REPO_ROOT = GENAI_ROOT
while REPO_ROOT.parent != REPO_ROOT and not (REPO_ROOT / '.git').exists():
    REPO_ROOT = REPO_ROOT.parent

def _disp_path(p):
    """Chemin affichable : relatif a la racine du depot quand il y est."""
    try:
        return Path(p).resolve().relative_to(REPO_ROOT.resolve())
    except ValueError:
        return Path(p).name

logging.basicConfig(level=getattr(logging, debug_level))
logger = logging.getLogger('transcription_pipeline')

print(f"Pipeline de Transcription et Sous-titrage")
print(f"Date : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Mode : {notebook_mode}")
print(f"Whisper : {'local (' + whisper_model + ')' if use_local_whisper else 'API (' + whisper_api_model + ')'}")
print(f"Sortie : {_disp_path(OUTPUT_DIR)}")

Helpers audio importes
Pipeline de Transcription et Sous-titrage
Date : 2026-11-12 14:37:42
Mode : interactive
Whisper : API (whisper-1)
Sortie : MyIA.AI.Notebooks\GenAI\outputs\audio\transcription


Les modules Python sont importes. Cette cellule charge les cles API depuis le fichier `.env` en remontant l'arborescence de repertoires pour garantir la compatibilite avec les différents environnements d'exécution.

In [4]:
# Chargement robuste de la configuration .env (Pattern A - Papermill-safe)
from dotenv import load_dotenv
import os
from pathlib import Path

current_path = Path.cwd()
genai_path = None

# Remonter jusqu'au dossier GenAI
while len(current_path.parts) > 1:
    if current_path.name == "GenAI":
        genai_path = current_path
        break
    current_path = current_path.parent

if genai_path:
    env_path = genai_path / ".env"
    if env_path.exists():
        load_dotenv(env_path)
        print(f".env charge depuis: {env_path.name}")

# Configuration API OpenAI
openai_key = os.environ.get('OPENAI_API_KEY', '')
openai_available = bool(openai_key and not openai_key.startswith('sk-or-'))

if openai_available:
    print("OPENAI_API_KEY valide - API disponible")
    from openai import OpenAI
    client = OpenAI(api_key=openai_key)
else:
    print("OPENAI_API_KEY non disponible ou invalide (cle OpenRouter)")
    openai_key = "dummy_key_for_validation"
    client = None

# Variables pour generation audio de test
tts_model = "tts-1"
narrator_voice = "nova"

# Generer des fichiers audio de test pour la transcription
test_audio_files = {}
if generate_audio and openai_available:
    print("\nGeneration de fichiers audio de test...")
    test_scripts = {
        "presentation": "Bonjour a tous, bienvenue dans cette presentation sur l'intelligence artificielle. Aujourd'hui, nous allons aborder trois sujets principaux, premierement les fondamentaux du machine learning, deuxiemement les reseaux de neurones, et troisiemement les applications pratiques.",
        "reunion": "Merci d'etre present pour cette reunion d'equipe. Le premier point a l'ordre du jour concerne les objectifs du trimestre. Nous devons finaliser le projet client avant la fin du mois."
    }
    
    for name, script in test_scripts.items():
        response = client.audio.speech.create(
            model=tts_model,
            voice=narrator_voice,
            input=script,
            response_format="mp3"
        )
        filepath = OUTPUT_DIR / f"test_{name}.mp3"
        with open(filepath, 'wb') as f:
            f.write(response.content)
        test_audio_files[name] = filepath
        print(f"  {name} : {filepath.name} ({len(response.content)/1024:.1f} KB)")

print("\nConfiguration prete")

.env charge depuis: .env
OPENAI_API_KEY valide - API disponible



Generation de fichiers audio de test...


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/audio/speech "HTTP/1.1 200 OK"


  presentation : test_presentation.mp3 (254.2 KB)


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/audio/speech "HTTP/1.1 200 OK"


  reunion : test_reunion.mp3 (171.8 KB)

Configuration prete


### Interpretation : Configuration de l'environnement

| Composant | Statut | Utilisation |
|-----------|--------|-------------|
| API OpenAI | Configuree | Transcription Whisper, synthese TTS, resume LLM |
| Helpers audio | Importes | Fonctions transcribe_openai, synthesize_openai |
| Repertoire sortie | Créé | `outputs/audio/transcription/` |
| Fichiers test | Generes (MP3) | presentation, reunion |

**Fichiers audio de test crees** :
- `test_presentation.mp3` : Expose de presentiel
- `test_reunion.mp3` : Simulation de reunion d'équipe

Ces fichiers serviront de base pour tous les tests du pipeline, garantissant la reproductibilite des demonstrations.

## Section 1 : Transcription avec horodatage

Une transcription de qualite necessite plus que le texte brut : les timestamps permettent de situer chaque segment dans le temps.

| Méthode | Latence | Qualite | Cout | Timestamps |
|---------|---------|---------|------|------------|
| Whisper API | ~10s/min | Bonne | $0.006/min | Segment-level |
| Whisper local (large-v3-turbo) | ~3s/min (GPU) | Excellente | Gratuit | Word-level |
| Whisper local (tiny) | ~0.5s/min (GPU) | Correcte | Gratuit | Segment-level |

Whisper local offre des timestamps au niveau du mot, essentiels pour les sous-titres précis.

***

## Introduction : Le pipeline de transcription

La transcription automatique de la parole (Automatic Speech Recognition, ASR) est la pierre angulaire de nombreuses applications IA audio. Mais une transcription de qualite ne se limite pas au texte brut : elle doit preserver la structure temporelle de l'enregistrement.

**Ce que vous allez apprendre** :
1. **Section 1** : Transcrire avec horodatage précis (timestamps)
2. **Section 2** : Automatiser le traitement de fichiers multiples
3. **Section 3** : Identifier les locuteurs (diarisation)
4. **Section 4** : Generer des sous-titres synchronises (SRT/VTT)
5. **Section 5** : Resumer automatiquement des reunions avec LLM

**Pourquoi les timestamps sont-ils essentiels ?**
- Sous-titrage video synchronise
- Navigation rapide dans l'audio
- Analyse temporelle (rythme, pauses, chevauchements)
- Indexation et recherche dans les enregistrements

Commencons par transcrire un fichier audio avec preservation des temps.

***

In [5]:
# Transcription avec horodatage
print("TRANSCRIPTION AVEC HORODATAGE")
print("=" * 50)

transcription_results = {}

if generate_audio and test_audio_files:
    test_file = test_audio_files.get("presentation")

    if test_file and test_file.exists():
        print(f"Fichier : {test_file.name}")

        if use_local_whisper and whisper_local_available:
            # Transcription locale avec timestamps
            print(f"\nMethode : Whisper local ({whisper_model})")
            start_time = time.time()

            try:
                model = WhisperModel(whisper_model, device=actual_whisper_device, compute_type=actual_compute_type)
                segments, info = model.transcribe(
                    str(test_file),
                    language=default_language,
                    word_timestamps=True
                )
                segments_list = list(segments)
                transcription_time = time.time() - start_time

                print(f"Langue detectee : {info.language} (prob: {info.language_probability:.2f})")
                print(f"Duree audio : {info.duration:.1f}s")
                print(f"Temps de transcription : {transcription_time:.1f}s")
                print(f"Ratio temps reel : {info.duration/transcription_time:.1f}x")

                print(f"\nSegments avec timestamps :")
                full_text = ""
                for seg in segments_list:
                    print(f"  [{seg.start:6.1f}s - {seg.end:6.1f}s] {seg.text.strip()}")
                    full_text += seg.text.strip() + " "

                transcription_results["local"] = {
                    "text": full_text.strip(),
                    "segments": segments_list,
                    "time": transcription_time,
                    "duration": info.duration
                }
            except Exception as e:
                print(f"Erreur transcription locale : {e}")
                print(f"Basculement vers l'API OpenAI")
                use_local_whisper = False

        if not use_local_whisper or not whisper_local_available or "local" not in transcription_results:
            # Transcription via API
            print(f"\nMethode : API OpenAI ({whisper_api_model})")
            start_time = time.time()

            with open(test_file, 'rb') as f:
                transcript = client.audio.transcriptions.create(
                    model=whisper_api_model,
                    file=f,
                    language=default_language,
                    response_format="verbose_json",
                    timestamp_granularities=["segment"]
                )
            transcription_time = time.time() - start_time

            print(f"Temps de transcription : {transcription_time:.1f}s")
            print(f"Texte : {transcript.text}")

            if hasattr(transcript, 'segments') and transcript.segments:
                print(f"\nSegments avec timestamps :")
                for seg in transcript.segments:
                    # TranscriptionSegment est un objet avec des attributs, pas subscriptable
                    start = getattr(seg, 'start', seg.get('start') if hasattr(seg, 'get') else 0)
                    end = getattr(seg, 'end', seg.get('end') if hasattr(seg, 'get') else 0)
                    text = getattr(seg, 'text', seg.get('text') if hasattr(seg, 'get') else '')
                    print(f"  [{start:6.1f}s - {end:6.1f}s] {text.strip()}")

            transcription_results["api"] = {
                "text": transcript.text,
                "time": transcription_time
            }

        # Ecouter le fichier original
        print(f"\nEcoute du fichier original :")
        display_audio_bundle(Path(test_file).read_bytes())
else:
    print("Transcription desactivee (pas de fichiers de test)")


TRANSCRIPTION AVEC HORODATAGE
Fichier : test_presentation.mp3

Methode : API OpenAI (whisper-1)


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/audio/transcriptions "HTTP/1.1 200 OK"


Temps de transcription : 1.5s
Texte : Bonjour à tous, bienvenue dans cette présentation sur l'intelligence artificielle. Aujourd'hui, nous allons aborder trois sujets principaux, premièrement les fondamentaux du machine learning, deuxièmement les réseaux de neurones et troisièmement les applications pratiques.

Segments avec timestamps :
  [   0.0s -    5.5s] Bonjour à tous, bienvenue dans cette présentation sur l'intelligence artificielle.
  [   5.5s -    8.6s] Aujourd'hui, nous allons aborder trois sujets principaux,
  [   8.6s -   11.7s] premièrement les fondamentaux du machine learning,
  [  11.7s -   16.2s] deuxièmement les réseaux de neurones et troisièmement les applications pratiques.

Ecoute du fichier original :


<audio embedded (audio/mpeg, 260352 bytes)>

### Interpretation : Transcription avec horodatage

| Aspect | Valeur | Signification |
|--------|--------|---------------|
| Ratio temps reel | >10x typique (GPU) | Transcription bien plus rapide que la duree audio |
| Precision timestamps | ~0.1s segments | Suffisant pour le sous-titrage standard |
| Detection langue | >95% probabilite | Whisper detecte fiablement le francais |

**Points cles** :
1. Les word-level timestamps ne sont disponibles qu'en mode local
2. L'API est plus simple mais offre moins de contrôle
3. Le ratio temps reel depend fortement du GPU et de la taille du modèle

## Section 2 : Transcription batch

En production, il est courant de transcrire de nombreux fichiers. Un pipeline batch doit gerer :

| Aspect | Description |
|--------|-------------|
| Decouverte | Lister les fichiers audio dans un repertoire |
| Filtrage | Verifier les formats supportes |
| Progression | Afficher l'avancement et les estimations de temps |
| Erreurs | Gerer les echecs sans bloquer le pipeline |
| Sortie | Sauvegarder les résultats de maniere structuree |

***

### Transition : Du fichier unique au pipeline batch

La Section 1 nous a permis de comprendre la transcription avec timestamps sur un seul fichier. En pratique, les besoins de transcription impliquent souvent des volumes importants : dizaines ou centaines de fichiers audio a traiter quotidiennement.

**Defis du traitement batch** :
- **Gestion des erreurs** : Un fichier corrompu ne doit pas bloquer tout le pipeline
- **Progression** : Afficher l'avancement pour les longs traitements
- **Performance** : Optimiser le chargement du modèle (une seule fois)
- **Sortie structuree** : Sauvegarder les résultats de facon exploitable

Cette section presente une architecture de pipeline batch robuste, reutilisable en production.

***

In [6]:
# Pipeline de transcription batch
print("TRANSCRIPTION BATCH")
print("=" * 50)

SUPPORTED_FORMATS = {'.mp3', '.wav', '.flac', '.ogg', '.m4a', '.webm'}

def batch_transcribe(input_dir: Path, output_dir: Path,
                     language: str = "fr") -> List[Dict[str, Any]]:
    """Transcrit tous les fichiers audio d'un repertoire."""
    results = []

    # Decouverte des fichiers
    audio_files = [
        f for f in sorted(input_dir.iterdir())
        if f.suffix.lower() in SUPPORTED_FORMATS
    ]

    if not audio_files:
        print("Aucun fichier audio trouve")
        return results

    print(f"Fichiers trouves : {len(audio_files)}")
    total_start = time.time()

    for i, audio_file in enumerate(audio_files):
        print(f"\n[{i+1}/{len(audio_files)}] {audio_file.name}")
        file_start = time.time()

        try:
            if use_local_whisper and whisper_local_available:
                model = WhisperModel(whisper_model, device=actual_whisper_device, compute_type=actual_compute_type)
                segments, info = model.transcribe(str(audio_file), language=language)
                segments_list = list(segments)
                text = " ".join([s.text.strip() for s in segments_list])
                duration = info.duration
            else:
                with open(audio_file, 'rb') as f:
                    transcript = client.audio.transcriptions.create(
                        model=whisper_api_model, file=f, language=language
                    )
                text = transcript.text
                duration = 0  # API ne retourne pas la duree en mode simple

            file_time = time.time() - file_start

            result = {
                "file": audio_file.name,
                "text": text,
                "duration": duration,
                "transcription_time": file_time,
                "word_count": len(text.split()),
                "status": "success"
            }
            results.append(result)

            # Sauvegarder la transcription
            if save_output_files:
                txt_path = output_dir / f"{audio_file.stem}.txt"
                txt_path.write_text(text, encoding='utf-8')

            print(f"  Duree : {duration:.1f}s | Temps : {file_time:.1f}s | Mots : {len(text.split())}")
            print(f"  Texte : {text[:80]}...")

        except Exception as e:
            results.append({
                "file": audio_file.name,
                "status": "error",
                "error": str(e)[:100]
            })
            print(f"  ERREUR : {str(e)[:80]}")

    total_time = time.time() - total_start
    success = sum(1 for r in results if r['status'] == 'success')
    print(f"\nBatch termine : {success}/{len(audio_files)} reussis en {total_time:.1f}s")

    return results

# Executer le batch
if generate_audio and test_audio_files:
    batch_results = batch_transcribe(OUTPUT_DIR, OUTPUT_DIR, language=default_language)

    # Recapitulatif
    if batch_results:
        print(f"\nRecapitulatif batch :")
        print(f"{'Fichier':<30} {'Statut':<10} {'Mots':<8} {'Temps (s)':<10}")
        print("-" * 58)
        for r in batch_results:
            if r['status'] == 'success':
                print(f"{r['file']:<30} {r['status']:<10} {r['word_count']:<8} {r['transcription_time']:<10.1f}")
            else:
                print(f"{r['file']:<30} {r['status']:<10} {'N/A':<8} {'N/A':<10}")
else:
    batch_results = []
    print("Batch desactivee (pas de fichiers de test)")


TRANSCRIPTION BATCH
Fichiers trouves : 2

[1/2] test_presentation.mp3


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/audio/transcriptions "HTTP/1.1 200 OK"


  Duree : 0.0s | Temps : 1.2s | Mots : 33
  Texte : Bonjour à tous, bienvenue dans cette présentation sur l'intelligence artificiell...

[2/2] test_reunion.mp3


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/audio/transcriptions "HTTP/1.1 200 OK"


  Duree : 0.0s | Temps : 1.0s | Mots : 30
  Texte : Merci d'être présent pour cette réunion d'équipe. Le premier point à l'ordre du ...

Batch termine : 2/2 reussis en 2.2s

Recapitulatif batch :
Fichier                        Statut     Mots     Temps (s) 
----------------------------------------------------------
test_presentation.mp3          success    33       1.2       
test_reunion.mp3               success    30       1.0       


### Interpretation : Transcription batch

| Aspect | Valeur | Signification |
|--------|--------|---------------|
| Debit batch | Variable selon GPU | L'initialisation du modèle est le goulot d'etranglement |
| Gestion erreurs | Try/except par fichier | Un echec ne bloque pas le pipeline |
| Sauvegarde | .txt par fichier | Format simple et universel |

> **Note technique** : En production, il est preferable de charger le modèle Whisper une seule fois et de transcrire tous les fichiers avec la même instance.

***

## Exercice : Pipeline de transcription avec gestion d'erreurs

**Duree estimee :** 20-25 minutes

### Objectif
Créer une fonction de transcription robuste qui gere différents formats audio, les erreurs API et les limites de taille de fichier.

### Contexte
En production, un pipeline de transcription doit gerer les fichiers de différentes tailles et formats, les erreurs reseau et les limites de l'API (taille max 25 MB pour Whisper). La robustesse du pipeline est aussi importante que sa precision.

### Instructions
1. Implementer une fonction qui accepte un chemin de fichier audio
2. Verifier le format et la taille avant envoi
3. Gerer les erreurs avec retries et fallback
4. Retourner un résultat structure avec statut et metadonnees

> **Indices :**
- `# Étape 1` : Verifier l'extension du fichier (.mp3, .wav, .flac, .m4a)
- `# Étape 2` : Verifier la taille (max 25 MB pour l'API OpenAI)
- `# Étape 3` : Appeler Whisper API avec `response_format="verbose_json"` pour les timestamps
- `# Indice` : Utiliser un try/except pour capturer les erreurs de timeout et de format
- `# Indice` : Implementer un retry avec `time.sleep()` en cas d'erreur transitoire

In [7]:
# TODO: Implementer une fonction de transcription robuste
SUPPORTED_FORMATS = {'.mp3', '.wav', '.flac', '.ogg', '.m4a', '.webm'}
MAX_FILE_SIZE_MB = 25

def robust_transcribe(audio_path: str, language: str = "fr",
                       max_retries: int = 2) -> dict:
    """
    Transcrit un fichier audio de maniere robuste avec validation et retries.
    
    Args:
        audio_path: Chemin vers le fichier audio
        language: Langue de l'audio
        max_retries: Nombre de tentatives en cas d'erreur
    
    Returns:
        Dict avec: text, segments, duration, status, error
    """
    # Etape 1 : Valider le fichier
    # Indice: Verifier extension et taille
    # Indice: Path(audio_path).suffix.lower() in SUPPORTED_FORMATS
    # Indice: Path(audio_path).stat().st_size / (1024*1024) <= MAX_FILE_SIZE_MB
    pass  # TODO etudiant
    
    # Etape 2 : Appeler l'API avec retry
    # Indice: Boucle for attempt in range(max_retries + 1):
    # Indice: try/except avec time.sleep(2 ** attempt) avant retry
    pass  # TODO etudiant
    
    # Etape 3 : Retourner le resultat structure
    # Indice: {"text": ..., "segments": [...], "status": "success"|"error", ...}
    return {"text": "", "segments": [], "status": "Exercice a completer"}

# Test avec un fichier de demonstration
# result = robust_transcribe("mon_audio.mp3", "fr")
# print(f"Statut : {result['status']}")
# print(f"Texte : {result['text'][:100]}...")
print("Exercice a completer")

Exercice a completer


## Section 3 : Diarisation de locuteurs

La diarisation identifie "qui parle quand" dans un enregistrement. C'est essentiel pour les reunions et interviews.

| Approche | Complexite | Precision | Prerequis |
|----------|-----------|-----------|----------|
| pyannote.audio | Elevee | Excellente | GPU, HuggingFace token |
| Energie simple | Faible | Basique | Aucun |
| Whisper segments | Moyenne | Correcte | Silences entre locuteurs |

Nous implementons ici une approche simplifiee basee sur l'analyse d'energie pour illustrer le concept.

***

### Transition : De la transcription a la comprehension contextuelle

Les deux premières sections nous ont permis de transcrire efficacement des fichiers audio, un par un (Section 1) ou en lot (Section 2). Cependant, une transcription brute ne preserve pas toujours le contexte de qui parle quand.

**Pourquoi la diarisation ?**
Dans une reunion ou un interview, identifier les locuteurs est essentiel pour :
- Attribuer correctement les decisions et actions
- Comprendre la dynamique de conversation
- Créer des comptes-rendus précis avec noms d'intervenants
- Analyser les temps de parole et la participation

**Approche progressive** : Nous commencons par une méthode simplifiee (energie du signal) avant d'aborder les techniques avancees (pyannote.audio).

***

In [8]:
# Diarisation simplifiee basee sur l'energie
print("DIARISATION DE LOCUTEURS")
print("=" * 50)

import numpy as np

# Verifier la disponibilite de pydub/ffmpeg
skip_diarization = False
try:
    from pydub import AudioSegment
    # Tester l'acces a ffmpeg
    test_segment = AudioSegment.silent(duration=100)
    del test_segment
except (ImportError, FileNotFoundError, OSError) as e:
    print(f"ATTENTION: pydub/ffmpeg non disponible - diarisation sautee")
    print(f"  Raison: {type(e).__name__}: {str(e)[:100]}")
    skip_diarization = True

def simple_energy_diarization(audio_path: str, window_ms: int = 500,
                               silence_threshold: float = 0.02) -> List[Dict]:
    """Diarisation simplifiee basee sur l'energie du signal.

    Detecte les segments de parole et les silences pour
    estimer les tours de parole.
    """
    from pydub import AudioSegment

    audio = AudioSegment.from_file(audio_path)
    samples = np.array(audio.get_array_of_samples(), dtype=np.float32)
    samples = samples / np.max(np.abs(samples))  # Normalisation

    sr = audio.frame_rate
    window_samples = int(sr * window_ms / 1000)

    # Calculer l'energie par fenetre
    segments = []
    is_speaking = False
    speaker_id = 0
    segment_start = 0

    for i in range(0, len(samples) - window_samples, window_samples):
        window = samples[i:i + window_samples]
        energy = np.sqrt(np.mean(window ** 2))  # RMS energy

        if energy > silence_threshold:
            if not is_speaking:
                segment_start = i / sr
                is_speaking = True
        else:
            if is_speaking:
                segment_end = i / sr
                # Nouveau segment de parole
                if segment_end - segment_start > 0.3:  # Min 300ms
                    segments.append({
                        "speaker": f"Speaker_{speaker_id % 2}",
                        "start": segment_start,
                        "end": segment_end,
                        "duration": segment_end - segment_start
                    })
                    speaker_id += 1
                is_speaking = False

    return segments

if not skip_diarization and generate_audio and test_audio_files:
    try:
        test_file = test_audio_files.get("reunion")

        if test_file and test_file.exists():
            print(f"Fichier : {test_file.name}")
            diarization_segments = simple_energy_diarization(str(test_file))

            print(f"\nSegments detectes : {len(diarization_segments)}")
            print(f"{'Locuteur':<15} {'Debut (s)':<12} {'Fin (s)':<12} {'Duree (s)':<10}")
            print("-" * 49)
            for seg in diarization_segments:
                print(f"{seg['speaker']:<15} {seg['start']:<12.1f} {seg['end']:<12.1f} {seg['duration']:<10.1f}")

            # Statistiques par locuteur
            speakers = set(s['speaker'] for s in diarization_segments)
            print(f"\nStatistiques par locuteur :")
            for speaker in sorted(speakers):
                speaker_segs = [s for s in diarization_segments if s['speaker'] == speaker]
                total_dur = sum(s['duration'] for s in speaker_segs)
                print(f"  {speaker} : {len(speaker_segs)} segments, {total_dur:.1f}s total")

            print(f"\n(Diarisation simplifiee - en production, utiliser pyannote.audio)")
        else:
            print("Fichier de test non disponible")
            diarization_segments = []
    except Exception as e:
        print(f"Erreur lors de la diarisation: {type(e).__name__}: {str(e)[:200]}")
        diarization_segments = []
else:
    diarization_segments = []
    if skip_diarization:
        print("Diarisation non disponible (ffmpeg requis)")
    else:
        print("Diarisation desactivee")


DIARISATION DE LOCUTEURS
Fichier : test_reunion.mp3



Segments detectes : 2
Locuteur        Debut (s)    Fin (s)      Duree (s) 
-------------------------------------------------
Speaker_0       0.0          2.5          2.5       
Speaker_1       3.0          7.0          4.0       

Statistiques par locuteur :
  Speaker_0 : 1 segments, 2.5s total
  Speaker_1 : 1 segments, 4.0s total

(Diarisation simplifiee - en production, utiliser pyannote.audio)


### Interpretation : Diarisation de locuteurs

| Aspect | Valeur observee | Signification |
|--------|----------------|---------------|
| Segments detectes | 2 (Speaker_0, Speaker_1) | Separation basee sur les silences |
| Precision temporelle | ~0.5s | Dependant du seuil d'energie |
| Alternance | Correcte | Model simplifie a 2 locuteurs |

**Limitations de l'approche energetique** :
1. Ne distingue pas vraiment les voix (seulement les silences)
2. Sensible au bruit de fond
3. Alternance arbitraire (Speaker_0/Speaker_1)

> **Note technique** : En production, utiliser **pyannote.audio** pour une diarisation vraie avec identification des locuteurs par embedding vocal.

## Section 4 : Generation de sous-titres SRT/VTT

Les formats de sous-titres standard permettent l'affichage synchronise avec la video :

| Format | Extension | Utilisation | Specifites |
|--------|-----------|-------------|------------|
| SRT (SubRip) | .srt | Universel | Simple, largement supporte |
| VTT (WebVTT) | .vtt | Web/HTML5 | Style CSS, regions, alignement |

### Format SRT
```
1
00:00:01,000 --> 00:00:04,500
Bienvenue dans cette presentation
```

### Format VTT
```
WEBVTT

00:00:01.000 --> 00:00:04.500
Bienvenue dans cette presentation
```

***

### Transition : Des timestamps aux sous-titres

La transcription avec timestamps (Section 1) nous donne une structure temporelle, mais elle n'est pas directement utilisable pour le sous-titrage video. Nous devons la convertir en formats standard (SRT/VTT) qui respectent les contraintes de lecture.

**Rappel des acquis précédents** :
- Section 1 : Transcription avec timestamps (segment-level)
- Section 2 : Pipeline batch pour traiter plusieurs fichiers
- Section 3 : Diarisation pour identifier les locuteurs

**Objectif de cette section** : Transformer les segments transcrits en sous-titres synchronises, utilisables dans les lecteurs video standard.

***

In [9]:
# Generation de sous-titres SRT et VTT
print("GENERATION DE SOUS-TITRES")
print("=" * 50)

def format_timestamp_srt(seconds: float) -> str:
    """Formate un timestamp en format SRT (HH:MM:SS,mmm)."""
    td = timedelta(seconds=seconds)
    hours = int(td.total_seconds() // 3600)
    minutes = int((td.total_seconds() % 3600) // 60)
    secs = int(td.total_seconds() % 60)
    ms = int((seconds % 1) * 1000)
    return f"{hours:02d}:{minutes:02d}:{secs:02d},{ms:03d}"

def format_timestamp_vtt(seconds: float) -> str:
    """Formate un timestamp en format VTT (HH:MM:SS.mmm)."""
    td = timedelta(seconds=seconds)
    hours = int(td.total_seconds() // 3600)
    minutes = int((td.total_seconds() % 3600) // 60)
    secs = int(td.total_seconds() % 60)
    ms = int((seconds % 1) * 1000)
    return f"{hours:02d}:{minutes:02d}:{secs:02d}.{ms:03d}"

def transcription_to_srt(segments: List[Dict], max_chars: int = 60) -> str:
    """Convertit des segments transcrits en format SRT."""
    srt_lines = []
    for i, seg in enumerate(segments, 1):
        start_ts = format_timestamp_srt(seg['start'])
        end_ts = format_timestamp_srt(seg['end'])
        text = seg.get('text', '').strip()
        # Couper les lignes longues
        if len(text) > max_chars:
            mid = len(text) // 2
            space_pos = text.rfind(' ', 0, mid)
            if space_pos > 0:
                text = text[:space_pos] + '\n' + text[space_pos+1:]
        srt_lines.append(f"{i}\n{start_ts} --> {end_ts}\n{text}\n")
    return '\n'.join(srt_lines)

def transcription_to_vtt(segments: List[Dict]) -> str:
    """Convertit des segments transcrits en format WebVTT."""
    vtt_lines = ["WEBVTT\n"]
    for seg in segments:
        start_ts = format_timestamp_vtt(seg['start'])
        end_ts = format_timestamp_vtt(seg['end'])
        text = seg.get('text', '').strip()
        vtt_lines.append(f"{start_ts} --> {end_ts}\n{text}\n")
    return '\n'.join(vtt_lines)

# Generer les sous-titres a partir de la transcription
if generate_audio and test_audio_files:
    test_file = test_audio_files.get("presentation")

    if test_file and test_file.exists():
        # Transcrire avec timestamps
        print(f"Transcription de {test_file.name} pour sous-titres...")

        if use_local_whisper and whisper_local_available:
            model = WhisperModel(whisper_model, device=actual_whisper_device, compute_type=actual_compute_type)
            segments, info = model.transcribe(str(test_file), language=default_language)
            sub_segments = [{"start": s.start, "end": s.end, "text": s.text.strip()} for s in segments]
        else:
            with open(test_file, 'rb') as f:
                transcript = client.audio.transcriptions.create(
                    model=whisper_api_model, file=f, language=default_language,
                    response_format="verbose_json", timestamp_granularities=["segment"]
                )
            sub_segments = [{"start": getattr(s, 'start', 0), "end": getattr(s, 'end', 0), "text": getattr(s, 'text', '').strip()}
                           for s in (transcript.segments or [])]

        if sub_segments:
            # Generer SRT
            srt_content = transcription_to_srt(sub_segments)
            print(f"\n--- Format SRT ---")
            print(srt_content[:500])

            # Generer VTT
            vtt_content = transcription_to_vtt(sub_segments)
            print(f"--- Format VTT ---")
            print(vtt_content[:500])

            # Sauvegarder
            if save_output_files:
                srt_path = OUTPUT_DIR / "presentation.srt"
                srt_path.write_text(srt_content, encoding='utf-8')
                vtt_path = OUTPUT_DIR / "presentation.vtt"
                vtt_path.write_text(vtt_content, encoding='utf-8')
                print(f"\nSauvegarde : {srt_path.name}, {vtt_path.name}")
        else:
            print("Aucun segment avec timestamps disponible")
else:
    print("Generation de sous-titres desactivee")


GENERATION DE SOUS-TITRES
Transcription de test_presentation.mp3 pour sous-titres...


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/audio/transcriptions "HTTP/1.1 200 OK"



--- Format SRT ---
1
00:00:00,000 --> 00:00:05,500
Bonjour à tous, bienvenue dans cette
présentation sur l'intelligence artificielle.

2
00:00:05,500 --> 00:00:08,600
Aujourd'hui, nous allons aborder trois sujets principaux,

3
00:00:08,600 --> 00:00:11,699
premièrement les fondamentaux du machine learning,

4
00:00:11,699 --> 00:00:16,200
deuxièmement les réseaux de neurones et
troisièmement les applications pratiques.

--- Format VTT ---
WEBVTT

00:00:00.000 --> 00:00:05.500
Bonjour à tous, bienvenue dans cette présentation sur l'intelligence artificielle.

00:00:05.500 --> 00:00:08.600
Aujourd'hui, nous allons aborder trois sujets principaux,

00:00:08.600 --> 00:00:11.699
premièrement les fondamentaux du machine learning,

00:00:11.699 --> 00:00:16.200
deuxièmement les réseaux de neurones et troisièmement les applications pratiques.


Sauvegarde : presentation.srt, presentation.vtt


### Interpretation : Generation de sous-titres

| Aspect | SRT | VTT |
|--------|-----|-----|
| Compatibilite | VLC, MPC, YouTube | Navigateurs web, HTML5 |
| Style | Aucun | CSS (couleur, position) |
| Separateur ms | Virgule (,) | Point (.) |
| Index | Obligatoire (1, 2, 3...) | Optionnel |

> **Note technique** : Pour des sous-titres de qualite broadcast, limiter chaque ligne a 42 caractères et chaque bloc a 2 lignes maximum.

***

## Exercice : Générateur de sous-titres optimises

**Duree estimee :** 20-25 minutes

### Objectif
Créer une fonction qui prend des segments transcrits avec timestamps et genere des sous-titres optimises respectant les règles broadcast (42 caractères par ligne, 2 lignes max, duree min 1s).

### Contexte
Les sous-titres de qualite broadcast doivent respecter des contraintes strictes de lisibilite. Le texte ne peut pas etre trop long, les sous-titres ne doivent pas rester trop peu de temps a l'ecran, et les segments trop longs doivent etre scindes intelligemment.

### Instructions
1. Implementer une fonction d'optimisation des segments
2. Scinder les segments depassant 42 caractères en multi-lignes
3. Fusionner les segments trop courts (< 1 seconde)
4. Verifier qu'aucun sous-titre ne depasse 5 secondes
5. Generer les formats SRT et VTT

> **Indices :**
- `# Étape 1` : Parcourir chaque segment et verifier sa longueur
- `# Étape 2` : Trouver l'espace le plus proche du milieu pour couper une ligne longue
- `# Étape 3` : Fusionner les segments adjacents de même locuteur si < 1s
- `# Indice` : Utiliser `text.rfind(' ', 0, max_chars // 2)` pour couper a un espace
- `# Indice` : Format SRT = index + timestamps + texte + ligne vide

In [10]:
# TODO: Implementer le generateur de sous-titres optimises
def optimize_subtitles(segments: list, max_chars_per_line: int = 42,
                       max_lines: int = 2, min_duration_s: float = 1.0,
                       max_duration_s: float = 5.0) -> list:
    """
    Optimise les segments de sous-titres pour la lisibilite broadcast.
    
    Args:
        segments: Liste de dicts {"start": float, "end": float, "text": str}
        max_chars_per_line: Maximum de caracteres par ligne
        max_lines: Maximum de lignes par sous-titre
        min_duration_s: Duree minimale d'un sous-titre en secondes
        max_duration_s: Duree maximale d'un sous-titre en secondes
    
    Returns:
        Liste de segments optimises
    """
    optimized = []
    
    for seg in segments:
        text = seg.get("text", "")
        duration = seg["end"] - seg["start"]
        
        # Etape 1 : Scinder le texte si trop long
        # Indice: Si len(text) > max_chars_per_line * max_lines, couper en morceaux
        pass  # TODO etudiant
        
        # Etape 2 : Gerer les segments trop courts
        # Indice: Si duration < min_duration_s, fusionner avec le precedent
        pass  # TODO etudiant
        
    return optimized  # TODO etudiant : retourner les segments optimises

# Test avec des segments de demonstration
demo_segments = [
    {"start": 0.0, "end": 3.5, "text": "Bienvenue dans cette presentation sur l'intelligence artificielle et ses applications"},
    {"start": 3.5, "end": 4.0, "text": "ici"},
    {"start": 4.0, "end": 8.0, "text": "Nous allons aborder trois sujets principaux aujourd'hui"},
]
# optimized = optimize_subtitles(demo_segments)
# for seg in optimized:
#     print(f"[{seg['start']:.1f}s - {seg['end']:.1f}s] {seg['text']}")
print("Exercice a completer")

Exercice a completer


## Section 5 : Resume automatique de reunions

Le pipeline complet pour resumer une reunion :

```
Audio reunion -> Whisper (transcription) -> GPT (resume) -> Document structure
```

Le LLM recoit la transcription brute et produit un resume structure avec points cles, decisions et actions.

***

### Transition : De la transcription a l'analyse

Après avoir maitrise la transcription avec timestamps (Section 1), le traitement batch (Section 2), la diarisation (Section 3) et les sous-titres (Section 4), nous allons maintenant exploiter ces données textuelles pour en extraire de la valeur ajoutée.

La transcription seule est une donnee brute. L'ajout d'un LLM permet de :
- **Resumer** des heures d'enregistrement en quelques minutes
- **Structurer** l'information (points cles, decisions, actions)
- **Extraire** des insights (sentiment, thèmes, sujets sensibles)
- **Generer** des documents automatiques (comptes-rendus, rapports)

***

In [11]:
# Resume automatique de reunions
print("RESUME AUTOMATIQUE DE REUNION")
print("=" * 50)

def summarize_meeting(transcript_text: str, llm_client, model: str) -> str:
    """Resume une transcription de reunion via LLM."""
    prompt = f"""
Analyse cette transcription de reunion et produis un resume structure :

1. RESUME (2-3 phrases)
2. POINTS CLES (liste a puces)
3. DECISIONS PRISES (liste numerotee)
4. ACTIONS A SUIVRE (tableau : Action | Responsable | Echeance)

Transcription :
{transcript_text}
"""
    response = llm_client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        max_completion_tokens=800
    )
    return response.choices[0].message.content

if generate_audio and test_audio_files:
    test_file = test_audio_files.get("reunion")

    if test_file and test_file.exists():
        # Transcrire la reunion
        print(f"Etape 1 : Transcription de {test_file.name}")
        start_time = time.time()

        if use_local_whisper and whisper_local_available:
            model = WhisperModel(whisper_model, device=actual_whisper_device, compute_type=actual_compute_type)
            segments, info = model.transcribe(str(test_file), language=default_language)
            meeting_text = " ".join([s.text.strip() for s in segments])
        else:
            with open(test_file, 'rb') as f:
                transcript = client.audio.transcriptions.create(
                    model=whisper_api_model, file=f, language=default_language
                )
            meeting_text = transcript.text

        transcription_time = time.time() - start_time
        print(f"Transcription : {len(meeting_text)} chars en {transcription_time:.1f}s")
        print(f"Texte : {meeting_text[:150]}...")

        # Resumer via LLM
        print(f"\nEtape 2 : Resume via {llm_model}")
        start_time = time.time()
        summary = summarize_meeting(meeting_text, client, llm_model)
        summary_time = time.time() - start_time

        print(f"Resume genere en {summary_time:.1f}s")
        print(f"\n--- Resume de la reunion ---")
        print(summary)
        print(f"--- Fin du resume ---")

        # Sauvegarder
        if save_output_files:
            summary_path = OUTPUT_DIR / "reunion_resume.md"
            summary_path.write_text(
                f"# Resume de reunion\n\n"
                f"Date : {datetime.now().strftime('%Y-%m-%d')}\n\n"
                f"{summary}\n\n"
                f"---\n\n"
                f"## Transcription complete\n\n{meeting_text}",
                encoding='utf-8'
            )
            print(f"\nSauvegarde : {summary_path.name}")
else:
    print("Resume desactive")


RESUME AUTOMATIQUE DE REUNION
Etape 1 : Transcription de test_reunion.mp3


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/audio/transcriptions "HTTP/1.1 200 OK"


Transcription : 182 chars en 1.1s
Texte : Merci d'être présent pour cette réunion d'équipe. Le premier point à l'ordre du jour concerne les objectifs du trimestre. Nous devons finaliser le pro...

Etape 2 : Resume via gpt-5.6-luna


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


Resume genere en 3.3s

--- Resume de la reunion ---
## 1. RÉSUMÉ

La réunion porte principalement sur les objectifs du trimestre. L’équipe doit finaliser le projet client avant la fin du mois.

## 2. POINTS CLÉS

- Revue des objectifs du trimestre.
- Priorité donnée à la finalisation du projet client.
- Échéance fixée à la fin du mois.

## 3. DÉCISIONS PRISES

1. Finaliser le projet client avant la fin du mois.

## 4. ACTIONS À SUIVRE

| Action | Responsable | Échéance |
|---|---|---|
| Finaliser le projet client | Équipe projet | Avant la fin du mois |
--- Fin du resume ---

Sauvegarde : reunion_resume.md


### Interpretation : Resume de reunion

| Étape | Temps typique | Cout (API) |
|-------|--------------|------------|
| Transcription (1h audio) | 30-60s (GPU local) | $0.36 (API) |
| Resume LLM | 3-5s | ~$0.01 (GPT-4o-mini) |
| Total pipeline | < 2 min | < $0.40 |

**Points cles** :
1. Le pipeline complet est realiste pour un usage quotidien
2. Le cout reste très faible même pour de longues reunions
3. La qualite du resume depend directement de la qualite de la transcription

***

## Exercice : Analyseur de reunion avec extraction d'actions

**Duree estimee :** 25-30 minutes

### Objectif
Créer un pipeline complet qui genere un fichier audio de reunion, le transcrit, et extrait automatiquement les actions a suivre avec echeances et responsables.

### Contexte
L'un des cas d'usage les plus valorissants de la transcription automatique est la generation de comptes-rendus de reunions avec actions assignees. Ce pipeline combine transcription (Whisper), analyse (LLM) et formatage structure.

### Instructions
1. Generer un audio de test simulant une reunion (plusieurs phrases avec actions)
2. Transcrire l'audio via l'API Whisper
3. Envoyer la transcription au LLM avec un prompt d'extraction structuree
4. Parser la reponse JSON pour extraire les actions
5. Afficher un tableau recapitulatif (Action, Responsable, Echeance)

> **Indices :**
- `# Étape 1` : Generer un audio avec TTS contenant au moins 3 phrases avec actions
- `# Étape 2` : Sauvegarder le fichier et appeler `client.audio.transcriptions.create()`
- `# Étape 3` : Prompt LLM : "Extrais les actions en JSON avec champs action, responsable, echeance"
- `# Indice` : Utiliser `json.loads()` pour parser la reponse du LLM
- `# Indice` : Demander au LLM de repondre UNIQUEMENT en JSON sans markdown

In [12]:
# TODO: Implementer le pipeline d'analyse de reunion
def analyze_meeting_actions(meeting_text: str = None, audio_path: str = None,
                             llm_client=None, model: str = "gpt-5.6-luna") -> list:
    """
    Pipeline complet : generation/transcription -> extraction d'actions.
    
    Args:
        meeting_text: Texte de reunion (si deja disponible)
        audio_path: Chemin audio (si transcription necessaire)
        llm_client: Client OpenAI
        model: Modele LLM
    
    Returns:
        Liste d'actions : [{"action": ..., "responsable": ..., "echeance": ...}]
    """
    # Etape 1 : Obtenir le texte de la reunion
    # Indice: Si meeting_text est None, transcrire audio_path via Whisper
    text = meeting_text  # TODO etudiant : transcrire si necessaire
    
    # Etape 2 : Extraire les actions via LLM
    # Indice: Prompt "Extrais les actions en JSON: [{action, responsable, echeance}]"
    # Indice: Demander "Reponds UNIQUEMENT en JSON, sans markdown ni commentaires"
    actions = []  # TODO etudiant
    
    # Etape 3 : Parser et retourner
    # Indice: json.loads() sur la reponse nettoyee
    return actions

# Test avec un texte de reunion de demonstration
demo_meeting = (
    "Bonjour a tous. Pierre, peux-tu finaliser le rapport financier pour vendredi prochain ? "
    "Marie, je te charge de preparer la presentation client pour lundi. "
    "L'equipe technique doit mettre a jour le serveur avant la fin du mois."
)
# actions = analyze_meeting_actions(demo_meeting, llm_client=client, model=llm_model)
# print(f"{'Action':<40} {'Responsable':<15} {'Echeance':<15}")
# print("-" * 70)
# for a in actions:
#     print(f"{a.get('action', '?'):<40} {a.get('responsable', '?'):<15} {a.get('echeance', '?'):<15}")
print("Exercice a completer")

Exercice a completer


In [13]:
# Mode interactif - Transcrire votre propre fichier audio
if notebook_mode == "interactive" and not skip_widgets:
    print("MODE INTERACTIF - TRANSCRIPTION PERSONNALISEE")
    print("=" * 50)
    print("\nEntrez le chemin d'un fichier audio a transcrire :")
    print("(Laissez vide pour passer)")

    try:
        user_path = input("\nChemin du fichier : ")

        if user_path.strip():
            audio_path = Path(user_path.strip())
            if audio_path.exists():
                print(f"\nTranscription de {audio_path.name}...")

                if use_local_whisper and whisper_local_available:
                    model = WhisperModel(whisper_model, device=actual_whisper_device, compute_type=actual_compute_type)
                    segments, info = model.transcribe(str(audio_path), language=default_language)
                    text = " ".join([s.text.strip() for s in segments])
                else:
                    with open(audio_path, 'rb') as f:
                        transcript = client.audio.transcriptions.create(
                            model=whisper_api_model, file=f, language=default_language
                        )
                    text = transcript.text

                print(f"\nTranscription :\n{text}")
                print(f"\nMots : {len(text.split())}")
            else:
                print(f"Fichier non trouve : {audio_path.name}")
        else:
            print("Mode interactif ignore")

    except (KeyboardInterrupt, EOFError):
        print("Mode interactif interrompu")
    except Exception as e:
        error_type = type(e).__name__
        if "StdinNotImplemented" in error_type or "input" in str(e).lower():
            print("Mode interactif non disponible (execution automatisee)")
        else:
            print(f"Erreur : {error_type} - {str(e)[:100]}")
else:
    print("Mode batch - Interface interactive desactivee")


MODE INTERACTIF - TRANSCRIPTION PERSONNALISEE

Entrez le chemin d'un fichier audio a transcrire :
(Laissez vide pour passer)
Mode interactif non disponible (execution automatisee)


Le module de statistiques consolide les metriques de la session : fichiers traites, durees de transcription, taux de confiance moyens et couts API cumules.

In [14]:
# Statistiques de session
print("STATISTIQUES DE SESSION")
print("=" * 50)

print(f"Date : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Mode : {notebook_mode}")
print(f"Whisper : {'local (' + whisper_model + ')' if use_local_whisper else 'API'}")
print(f"LLM resume : {llm_model}")

if transcription_results:
    for method, data in transcription_results.items():
        print(f"Transcription ({method}) : {data['time']:.1f}s")

if batch_results:
    success = sum(1 for r in batch_results if r['status'] == 'success')
    print(f"Batch : {success}/{len(batch_results)} fichiers transcrits")

if save_output_files:
    saved_files = list(OUTPUT_DIR.glob('*'))
    print(f"Fichiers sauvegardes : {len(saved_files)} dans {_disp_path(OUTPUT_DIR)}")

print(f"\nPROCHAINES ETAPES")
print(f"1. Explorer la composition musicale (04-3-Music-Composition-Workflow)")
print(f"2. Synchroniser audio et video (04-4-Audio-Video-Sync)")

print(f"\nNotebook termine - {datetime.now().strftime('%H:%M:%S')}")

STATISTIQUES DE SESSION
Date : 2026-11-12 14:37:59
Mode : interactive
Whisper : API
LLM resume : gpt-5.6-luna
Transcription (api) : 1.5s
Batch : 2/2 fichiers transcrits
Fichiers sauvegardes : 7 dans MyIA.AI.Notebooks\GenAI\outputs\audio\transcription

PROCHAINES ETAPES
1. Explorer la composition musicale (04-3-Music-Composition-Workflow)
2. Synchroniser audio et video (04-4-Audio-Video-Sync)

Notebook termine - 14:37:59


***

## Exemple guide : Pipeline de Sous-Titrage Video

**Duree estimee :** 25-30 minutes

### Objectif
Créer un pipeline complet de sous-titrage pour une video : transcription, generation de sous-titres SRT/VTT, et synchronisation.

### Instructions
1. Extraire l'audio d'une video (ou utiliser un fichier audio de test)
2. Transcrire l'audio avec timestamps précis
3. Generer des sous-titres aux formats SRT et VTT
4. Optimiser les sous-titres (longueur des lignes, timing)
5. Synchroniser et verifier la qualite

> **Indices :**
- Pour extraire l'audio video: utiliser `moviepy` ou `ffmpeg`
- Pour la transcription: Whisper API avec `response_format="verbose_json"`
- Pour l'optimisation: limiter a 42 caractères par ligne, 2 lignes max
- Pour la synchronisation: verifier que les sous-titres ne se chevauchent pas

## Mise en pratique

Le pipeline de transcription que nous avons construit peut etre etendu a de nombreux cas d'usage reels :

**Applications professionnelles** :
- Sous-titrage automatique de videos YouTube/LinkedIn
- Comptes-rendus de reunions automatiques
- Transcription de podcasts pour le SEO
- Accessibilite (handicap auditif)

**Defis techniques** :
- Qualite audio variable (bruit, eco, accents)
- Termes techniques et jargon spécifique
- Locuteurs multiples et chevauchements
- Volume important (traitement batch)

L'exercice suivant vous guide dans la creation d'un pipeline complet de sous-titrage video, en integrant tous les concepts abordés.

In [15]:
# Exercice: Fonction d'extraction audio depuis une video
def extract_audio_from_video(video_path: str, output_audio_path: str) -> bool:
    """
    Extrait la piste audio d'une video.
    
    Args:
        video_path: Chemin vers le fichier video
        output_audio_path: Chemin de sortie pour l'audio (WAV/MP3)
    
    Returns:
        True si succes, False sinon
    """
    # Indice: Utiliser moviepy.VideoFileClip pour extraire l'audio
    # Indice: video_clip = VideoFileClip(video_path)
    # Indice: audio_clip = video_clip.audio
    # Indice: audio_clip.write_audiofile(output_audio_path)
    pass

# Exercice: Fonction de transcription avec timestamps
def transcribe_with_timestamps(audio_path: str, language: str = "fr") -> list:
    """
    Transcrit l'audio avec timestamps precis.
    
    Args:
        audio_path: Chemin vers le fichier audio
        language: Langue de l'audio
    
    Returns:
        Liste de segments: [{"start": 0.0, "end": 2.5, "text": "..."}]
    """
    # Indice: Appeler Whisper API avec response_format="verbose_json"
    # Indice: Parser les resultats pour extraire start, end, text
    pass

# (optimize_subtitles est l'objet de l'Exercice "Generateur de sous-titres optimises" ci-dessus -- non redefini ici.)

# Exercice: Fonction de generation SRT
def generate_srt(segments: list, output_path: str) -> None:
    """
    Genere un fichier de sous-titres au format SRT.
    
    Args:
        segments: Liste de segments optimises
        output_path: Chemin du fichier SRT de sortie
    """
    # Indice: Utiliser le format SRT (voir notebook)
    # Indice: format_timestamp_srt() du notebook
    pass

# Exercice: Pipeline complet
# audio = extract_audio_from_video("ma_video.mp4", "audio_temp.wav")
# segments = transcribe_with_timestamps("audio_temp.wav", "fr")
# optimized = optimize_subtitles(segments)
# generate_srt(optimized, "sous_titres.srt")

### Critères de succès
- [ ] Fonction d'extraction audio fonctionnelle
- [ ] Transcription avec timestamps précis
- [ ] Sous-titres optimises (longueur, timing)
- [ ] Fichiers SRT et VTT generes
- [ ] Verification de synchronisation

### Extension (optionnel)
- [ ] Ajouter la detection de langues automatique
- [ ] Créer une interface de correction manuelle
- [ ] Generer des sous-titres multilingues
- [ ] Integrer la diarisation de locuteurs

## Synthese du Pipeline de Transcription

Ce notebook a presente un pipeline complet de transcription audio, de l'enregistrement brut au document structure.

| Composant | Rôle | Outil recommande |
|-----------|------|------------------|
| Transcription | Audio → texte avec timestamps | Whisper (local/API) |
| Diarisation | Identification des locuteurs | pyannote.audio |
| Sous-titrage | Synchronisation video | SRT/VTT |
| Resumption | Synthese automatique | LLM (GPT-4o-mini) |

**Bonnes pratiques** :
- Toujours conserver les timestamps pour la post-production
- Valider la qualite de transcription sur un echantillon
- Optimiser les sous-titres (42 caractères/ligne, 2 lignes max)
- Documenter les paramètres du pipeline pour la reproductibilite

**Prochaines étapes** :
1. Explorer la composition musicale avec IA (notebook 04-3)
2. Synchroniser audio et video pour le montage automatique (notebook 04-4)

## Conclusion

Ce notebook a permis d'explorer les aspects essentiels de 04 2 transcription pipeline. Les points cles :

- Les concepts fondamentaux ont ete presentes et illustres
- Les exercices proposent une mise en pratique progressive
- Les résultats obtenus permettent de valider la comprehension

**Pour aller plus loin** : approfondir les aspects avances du sujet et explorer les liens avec d'autres domaines.